# EMA Crossover Strategy — Backtest Research

**Instrument:** QQQ (5-min bars)
**Signal:** Triple EMA alignment (13 / 48 / 200)
**Exit:** ATR-based or percentage-based stop loss, or EOD (15:55 ET)
**Position sizing:** Risk-based with configurable leverage

Two strategy variants are tested:
- **Strategy 1** — EMA crossover + price must exceed previous day / premarket threshold
- **Strategy 2** — EMA crossover only (no threshold filter)

Uses `shared/` for data fetching, fees, metrics, and plotting.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")  # so we can import shared/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

from _shared.loaders_data import fetch_historical_data
from _shared.fees import calculate_fees
from _shared.metrics import evaluate_strategy, print_metrics, compare_strategies
from _shared.plotting import plot_equity_curve, plot_equity_comparison, plot_trade_returns, plot_yearly_returns
from _shared.results import save_trades, load_trades

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

SYMBOL     = "QQQ"
START_DATE = "2016-01-01"
END_DATE   = "2026-04-01"

# Backtest parameters
STARTING_CAPITAL = 100_000
LEVERAGE         = 1
RISK_PERCENT     = 0.01   # 1% risk per trade
MAX_CAP_PERCENT  = 1      # max 100% of capital per trade

# Fee toggle
INCLUDE_FEES = True

## 2. Data Fetching

In [ ]:
# Fetch 5-minute and daily data via shared fetcher
data_5min = fetch_historical_data(
    [SYMBOL], TimeFrame(5, TimeFrameUnit.Minute), START_DATE, END_DATE)
data_minute = data_5min[SYMBOL]

data_1d = fetch_historical_data(
    [SYMBOL], TimeFrame(1, TimeFrameUnit.Day), START_DATE, END_DATE)
data_daily = data_1d[SYMBOL]

print(f"\n5-min bars: {len(data_minute):,}")
print(f"Daily bars: {len(data_daily):,}")

## 3. Feature Engineering

### 3a. EMAs

In [ ]:
data_minute["ema13"]  = data_minute["close"].ewm(span=13,  adjust=False).mean()
data_minute["ema48"]  = data_minute["close"].ewm(span=48,  adjust=False).mean()
data_minute["ema200"] = data_minute["close"].ewm(span=200, adjust=False).mean()
print("EMAs calculated.")

### 3b. ATR-14

In [ ]:
prev_close = data_daily["close"].shift(1)
tr = pd.concat([
    data_daily["high"] - data_daily["low"],
    (data_daily["high"] - prev_close).abs(),
    (data_daily["low"]  - prev_close).abs(),
], axis=1).max(axis=1)

atr14_prev = tr.ewm(span=14, adjust=False).mean().shift(1)  # previous day — no look-ahead
atr_df = atr14_prev.to_frame(name="atr14")
atr_df["date"] = pd.to_datetime(atr_df.index.date)
print("ATR-14 calculated.")

### 3c. Merge & previous-day levels

In [ ]:
data = data_minute.copy()
data["date"] = pd.to_datetime(data.index.date)

original_index = data.index
data = data.merge(atr_df[["date", "atr14"]], on="date", how="left")
data.index = original_index

# Previous day high/low
prev_day_high = data.groupby("date")["high"].max()
prev_day_low  = data.groupby("date")["low"].min()

dates = pd.Series(data["date"].unique()).sort_values()
prev_date_map = dict(zip(dates[1:], dates[:-1]))

data["prev_date"]     = data["date"].map(prev_date_map)
data["prev_day_high"] = data["prev_date"].map(prev_day_high)
data["prev_day_low"]  = data["prev_date"].map(prev_day_low)

# Premarket extremes
market_open = pd.to_datetime("09:30:00").time()
data = data.tz_convert("America/New_York")
premarket = data[data.index.time < market_open].copy()
premarket_extremes = premarket.groupby("date").agg({"high": "max", "low": "min"})
premarket_extremes.columns = ["premarket_high", "premarket_low"]
data = data.join(premarket_extremes, on="date")

data["threshold_high"] = data[["prev_day_high", "premarket_high"]].max(axis=1)
data["threshold_low"]  = data[["prev_day_low",  "premarket_low"]].min(axis=1)

print(f"Merged data shape: {data.shape}")

## 4. Signal Generation

In [ ]:
data = data.dropna(subset=["atr14", "ema13", "ema48", "ema200", "threshold_high", "threshold_low"])

market_open  = pd.to_datetime("09:30:00").time()
market_close = pd.to_datetime("16:00:00").time()
data["regular_session"] = (data.index.time >= market_open) & (data.index.time < market_close)

# Strategy 1: EMA crossover + threshold
data_strat1 = data[data["regular_session"]].copy()
l1 = (data_strat1["ema13"] > data_strat1["ema48"]) & (data_strat1["ema48"] > data_strat1["ema200"]) & (data_strat1["close"] > data_strat1["threshold_high"])
s1 = (data_strat1["ema13"] < data_strat1["ema48"]) & (data_strat1["ema48"] < data_strat1["ema200"]) & (data_strat1["close"] < data_strat1["threshold_low"])
data_strat1["long_entries"]  = l1.shift(1).fillna(False)
data_strat1["short_entries"] = s1.shift(1).fillna(False)

# Strategy 2: EMA crossover only
data_strat2 = data[data["regular_session"]].copy()
l2 = (data_strat2["ema13"] > data_strat2["ema48"]) & (data_strat2["ema48"] > data_strat2["ema200"])
s2 = (data_strat2["ema13"] < data_strat2["ema48"]) & (data_strat2["ema48"] < data_strat2["ema200"])
data_strat2["long_entries"]  = l2.shift(1).fillna(False)
data_strat2["short_entries"] = s2.shift(1).fillna(False)

print(f"Strategy 1 — L: {data_strat1['long_entries'].sum():,}, S: {data_strat1['short_entries'].sum():,}")
print(f"Strategy 2 — L: {data_strat2['long_entries'].sum():,}, S: {data_strat2['short_entries'].sum():,}")

## 5. Backtest Engine

Strategy-specific loop, but outputs the **standard trades DataFrame** that
`shared.metrics` and `shared.plotting` expect.

In [ ]:
def run_backtest(data_regular, stop_type="atr", stop_value=0.05,
                 starting_capital=STARTING_CAPITAL, include_fees=INCLUDE_FEES):
    """
    Run EMA crossover backtest, returning a standardized trades DataFrame.

    Parameters
    ----------
    data_regular : pd.DataFrame — regular-session bars with signals and atr14.
    stop_type    : str   — 'atr' or 'pct'.
    stop_value   : float — stop multiplier.
    starting_capital : float
    include_fees : bool

    Returns
    -------
    pd.DataFrame with standard columns (see shared.metrics.REQUIRED_COLUMNS).
    """
    trades = []
    holding = False

    for timestamp, row in data_regular.iterrows():
        if holding:
            # Stop loss
            if entry_type == "long" and row["low"] <= stop_price:
                exit_price, exit_time, exit_reason = stop_price, timestamp, "stop"
                holding = False
            elif entry_type == "short" and row["high"] >= stop_price:
                exit_price, exit_time, exit_reason = stop_price, timestamp, "stop"
                holding = False
            elif timestamp.hour == 15 and timestamp.minute >= 55:
                exit_price, exit_time, exit_reason = row["close"], timestamp, "eod"
                holding = False
            else:
                continue

            # Record trade
            if entry_type == "long":
                gross = shares * (exit_price - entry_price)
            else:
                gross = shares * (entry_price - exit_price)

            fees = calculate_fees(shares, entry_price, exit_price, entry_type) if include_fees else 0.0
            net = gross - fees
            equity += net

            trades.append({
                "entry_time":   entry_time,
                "exit_time":    exit_time,
                "position":     entry_type,
                "entry_price":  entry_price,
                "exit_price":   exit_price,
                "exit_reason":  exit_reason,
                "risk":         risk,
                "shares":       shares,
                "gross_pnl":    gross,
                "fees":         fees,
                "net_pnl":      net,
                "equity_before": eq_before,
                "equity":       equity,
            })

            if equity <= 0:
                print(f"Account blew up at trade {len(trades)}")
                break
            continue

        # Entry signals
        if row.get("long_entries", False) or row.get("short_entries", False):
            entry_type = "long" if row.get("long_entries", False) else "short"
            entry_price = row["open"]
            entry_time = timestamp

            if stop_type == "atr":
                risk = row["atr14"] * stop_value
            else:
                risk = entry_price * stop_value

            if entry_type == "long":
                stop_price = entry_price - risk
            else:
                stop_price = entry_price + risk

            # Position sizing
            eq_before = equity if trades else starting_capital
            if not trades:
                equity = starting_capital

            eq_before = equity
            if risk <= 0 or equity <= 0:
                continue

            shares_by_risk = (equity * RISK_PERCENT) / risk
            shares_by_lev  = (equity * LEVERAGE) / entry_price
            shares = int(min(shares_by_risk, shares_by_lev))

            cap_used = (shares * entry_price) / LEVERAGE
            if cap_used > equity * MAX_CAP_PERCENT:
                shares = int(equity * MAX_CAP_PERCENT * LEVERAGE / entry_price)

            if shares <= 0:
                continue

            holding = True

    if not trades:
        return pd.DataFrame()

    result = pd.DataFrame(trades)
    result["duration_hours"] = (result["exit_time"] - result["entry_time"]).dt.total_seconds() / 3600
    return result

# Initialize equity tracker
equity = STARTING_CAPITAL

## 6. Run Backtest — Strategy Comparison

In [ ]:
equity = STARTING_CAPITAL
results_1 = run_backtest(data_strat1, stop_type="atr", stop_value=0.05)

equity = STARTING_CAPITAL
results_2 = run_backtest(data_strat2, stop_type="atr", stop_value=0.05)

print(f"Strategy 1: {len(results_1):,} trades")
print(f"Strategy 2: {len(results_2):,} trades")

In [ ]:
# Compare using shared metrics
m1 = evaluate_strategy(results_1, "Strategy 1 (Threshold)")
m2 = evaluate_strategy(results_2, "Strategy 2 (EMA Only)")

comparison = compare_strategies([m1, m2])
print(comparison)

In [ ]:
# Equity curves via shared plotting
plot_equity_comparison(
    [results_1, results_2],
    ["Strategy 1: Threshold", "Strategy 2: EMA Only"],
    STARTING_CAPITAL,
    title="EMA Crossover — Strategy Comparison",
)

## 7. Stop-Loss Comparison (Strategy 2)

In [ ]:
configs = [
    ("ATR 0.03x", "atr", 0.03),
    ("ATR 0.05x", "atr", 0.05),
    ("PCT 0.030%", "pct", 0.0003),
    ("PCT 0.050%", "pct", 0.0005),
]

stop_metrics = []
stop_curves = []
stop_labels = []

for label, stype, sval in configs:
    global equity
    equity = STARTING_CAPITAL
    res = run_backtest(data_strat2, stop_type=stype, stop_value=sval, include_fees=False)
    if res.empty:
        continue
    stop_metrics.append(evaluate_strategy(res, label))
    stop_curves.append(res)
    stop_labels.append(label)

print(compare_strategies(stop_metrics))

plot_equity_comparison(stop_curves, stop_labels, STARTING_CAPITAL,
                       title="Strategy 2 — Stop-Loss Comparison (NO FEES)")

> **Conclusion:** Strategy 2 with ATR 0.05x stop preferred. Proceed with fees.

## 8. Final Backtest — Strategy 2 with Fees

In [ ]:
equity = STARTING_CAPITAL
results_final = run_backtest(data_strat2, stop_type="atr", stop_value=0.05, include_fees=True)

# Shared metrics
metrics_final = evaluate_strategy(results_final, "Strategy 2 (with fees)")
print_metrics(metrics_final)

# Shared plots
plot_equity_curve(results_final, label="Strategy 2 (with fees)",
                  starting_capital=STARTING_CAPITAL)
plot_trade_returns(results_final, title="Strategy 2 — Per-Trade Returns")
plot_yearly_returns(metrics_final, title="Strategy 2 — Yearly Returns")

## 9. Save Results

In [ ]:
save_trades(results_final, "ema_crossover")

---

## Research Filters (Optional)

The sections below are exploratory — trade-churn limits, time-of-day filters,
and ATR-regime filters. These use strategy-specific columns beyond the standard
format and are kept here for reference.

*Note: The research filter code from the previous version can be appended here
as needed. It operates on `results_final` and `data_strat2` which are available
at this point in the notebook.*